Star Schema for Olist Dataset
Fact Table : gold_fact_sales
Aggregated Buisness Marts : gold_daily_sales_summary,  gold_seller_performance_metrics, gold_customer_ltv_analytics

In [0]:
from delta.tables import DeltaTable
import os 
import sys
from pyspark.sql import functions as F

# ==========================================
# 1. EXTRACT SILVER TABLES
# ==========================================
print("Extracting Silver tables for Gold fact Sales...")

df_items = spark.read.table("ecommerce_lakehouse.silver.silver_order_items")
df_orders = spark.read.table("ecommerce_lakehouse.silver.silver_orders")
df_products = spark.read.table("ecommerce_lakehouse.silver.silver_products")
df_sellers = spark.read.table("ecommerce_lakehouse.silver.silver_sellers")
df_payments = spark.read.table("ecommerce_lakehouse.silver.silver_order_payments")

# ==========================================
# 2. AGGREGATE PAYMENTS (Handle split payments per order)
# ==========================================
# An order can have multiple payment methods; we aggregate to order-level for the fact table mapping

df_payments_agg = df_payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("total_payment_value"),
    F.first("payment_type").alias("primary_payment_type"),
    F.max("payment_installments").alias("max_installments"),
)

# ==========================================
# 3. BUILD FACT TABLE JOINS & TRANSFORMATIONS
# ==========================================
df_fact = (
    df_items.alias("i")
    .join(df_orders.alias("o"), on="order_id", how='inner')
    .join(df_products.alias("p"), on='product_id', how='left')
    .join(df_sellers.alias("s"), on='seller_id', how="left")
    .join(df_payments_agg.alias("pay"), on="order_id", how='left')
)
# Select and derive analytical attributes
df_transformed = df_fact.select(
    #id's and surrogate key's
    F.col("i.order_id"),
    F.col("i.order_item_id"),
    F.coalesce(F.col("o.customer_key"),F.lit("C_UNKNOWN")).alias("customer_key"),
    F.coalesce(F.col("i.product_key"), F.lit("P_UNKNOWN")).alias("product_key"),
    F.coalesce(F.col("i.seller_key"), F.lit("S_UNKNOWN")).alias("seller_key"),
    #Timestamps and derived date attributes
    F.col("o.purchase_ts"),
    F.to_date(F.col("o.purchase_ts")).alias("purchase_date"),
    F.year(F.col("o.purchase_ts")).alias("purchase_year"),
    F.month(F.col("o.purchase_ts")).alias("purchase_month"),
    F.date_format(F.col("o.purchase_ts"),"EEEE").alias("purchase_day_of_week"),
    F.col("o.approved_ts"),
    F.col("o.delivered_carrier_ts"),
    F.col("o.delivered_customer_ts"),
    F.col("o.estimated_delivery_ts"),
    #order & Item Status
    F.col("o.order_status"),
    F.col("p.category_name"),
    F.col("s.seller_state"),
    F.col("pay.primary_payment_type"),
    F.col("pay.max_installments"),
    # Financial Metrics
    F.col("i.price"),
    F.col("i.freight_value"),
    (F.col("i.price")+F.col("i.freight_value")).alias("total_item_amount"),
    F.col("pay.total_payment_value"),
    #Logistics Performance metrics
    F.when(
        F.col("o.delivered_customer_ts")>F.col("o.estimated_delivery_ts"),True,
    ).otherwise(False).alias("is_delayed"),
    F.datediff(
        F.col("o.delivered_customer_ts"), F.col("o.purchase_ts")
    ).alias("fulfillment_days"),
).dropDuplicates(["order_id", "order_item_id"])

# ==========================================
# 4. WRITE TO GOLD LAYER (Overwrite/Append model for Fact)
# ==========================================
target_table_name = "ecommerce_lakehouse.gold.gold_fact_sales"
print(f"Writing data to Gold table: {target_table_name}")

df_transformed.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(target_table_name)

print("Successfully generated gold_fact_sales!")

Extracting Silver tables for Gold fact Sales...
Writing data to Gold table: ecommerce_lakehouse.gold.gold_fact_sales
Successfully generated gold_fact_sales!
